# Document Question Answering System (RAG)
### Week 7 Assignment - Retrieval-Augmented Generation

This notebook builds a complete RAG pipeline:
1. Document Ingestion (PDF/TXT)
2. Text Chunking
3. Embedding Creation
4. Vector Database (FAISS)
5. Query Processing
6. Context Retrieval
7. Answer Generation (LLM)

**Author:** Mahesh Shinde  
**Reference:** https://github.com/VivekChauhan05/RAG_Document_Question_Answering

In [13]:
!pip install -q pypdf sentence-transformers faiss-cpu transformers accelerate torch langchain langchain-community tiktoken

## 2. Imports & Config

In [14]:
import os
import re
import glob
import numpy as np
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# ---- Config ----
DOCS_FOLDER = "documents"          # put your PDFs / .txt files here
CHUNK_SIZE = 500                    # characters per chunk
CHUNK_OVERLAP = 100                 # overlap between chunks
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"          # fast, small, good quality
GEN_MODEL_NAME = "google/flan-t5-base"          # free, local generation model
TOP_K = 4                            # number of chunks to retrieve

os.makedirs(DOCS_FOLDER, exist_ok=True)
print(f"Put your PDF/TXT files inside the '{DOCS_FOLDER}/' folder, then re-run the ingestion cell.")

Put your PDF/TXT files inside the 'documents/' folder, then re-run the ingestion cell.


## 3. Document Ingestion
Loads all PDFs and text files from `DOCS_FOLDER` and extracts raw text.

In [15]:
def load_pdf(path):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text() or ""
        text += page_text + "\n"
    return text

def load_txt(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def load_documents(folder):
    docs = []  # list of (filename, text)
    for path in glob.glob(os.path.join(folder, "*")):
        if path.lower().endswith(".pdf"):
            text = load_pdf(path)
        elif path.lower().endswith(".txt"):
            text = load_txt(path)
        else:
            continue
        text = re.sub(r"\s+", " ", text).strip()
        if text:
            docs.append((os.path.basename(path), text))
    return docs

documents = load_documents(DOCS_FOLDER)
print(f"Loaded {len(documents)} document(s).")
for name, text in documents:
    print(f" - {name}: {len(text)} characters")

Loaded 2 document(s).
 - sample_rag_notes.txt: 1600 characters
 - resume.pdf: 2834 characters


### 3b. Sample fallback text (use this if you don't have a PDF handy)
This creates a demo `.txt` file so the notebook is fully runnable end-to-end without any upload.

In [16]:
sample_text = """
Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with
text generation. Instead of relying purely on a language model's parametric memory, RAG systems
first search an external knowledge base for relevant passages, then feed those passages to the
language model as context so the generated answer is grounded in real, verifiable data.

A typical RAG pipeline has three main stages: retrieval, augmentation, and generation. In the
retrieval stage, the user's query is converted into a vector embedding and compared against a
vector database of document chunk embeddings using similarity search (commonly cosine similarity
or L2 distance). In the augmentation stage, the top matching chunks are inserted into the model's
prompt. In the generation stage, a language model produces the final answer using both the
original question and the retrieved context.

RAG systems are useful because they reduce hallucination, allow question answering over private
or domain-specific documents, and avoid the need to retrain the model whenever the underlying
knowledge changes. Common use cases include chatbots, enterprise search, customer support
assistants, and research paper summarization tools.

Popular vector databases include FAISS, Chroma, Pinecone, and Weaviate. Popular embedding models
include Sentence-BERT variants such as all-MiniLM-L6-v2, and larger models like OpenAI's
text-embedding-3. Chunking strategy (chunk size and overlap) has a large impact on retrieval
quality: chunks that are too large dilute relevance, while chunks that are too small lose context.
"""

sample_path = os.path.join(DOCS_FOLDER, "sample_rag_notes.txt")
with open(sample_path, "w") as f:
    f.write(sample_text)

# reload documents including the sample file
documents = load_documents(DOCS_FOLDER)
print(f"Loaded {len(documents)} document(s) (including sample).")

Loaded 2 document(s) (including sample).


## 4. Text Chunking
Splits each document into overlapping chunks so retrieval can pinpoint relevant passages.

In [34]:
all_chunks = []
chunk_metadata = []

for filename, text in documents:
    chunks = chunk_text(text)
    for i, c in enumerate(chunks):
        all_chunks.append(c)
        chunk_metadata.append({"source": filename, "chunk_id": i})

print(f"Total chunks created: {len(all_chunks)}")

chunk_embeddings = embed_model.encode(
    all_chunks,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embedding_dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.add(chunk_embeddings)
print(f"FAISS index rebuilt with {index.ntotal} vectors.")

Total chunks created: 8


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS index rebuilt with 8 vectors.


In [36]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

GEN_MODEL_NAME = "google/flan-t5-large"
print("Loading flan-t5-large (bigger download, give it a minute)...")
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL_NAME)

def build_prompt(question, context_chunks):
    context_text = "\n\n".join([c["text"] for c in context_chunks])
    prompt = (
        f"Context:\n{context_text}\n\n"
        f"Based on the context above, answer this question as specifically as possible: {question}\n"
        "Answer:"
    )
    return prompt

def generate_answer(question, top_k=TOP_K):
    context_chunks = retrieve_context(question, top_k=top_k)
    prompt = build_prompt(question, context_chunks)
    input_ids = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).input_ids
    output_ids = gen_model.generate(input_ids, max_new_tokens=200)
    answer = gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return answer, context_chunks

Loading flan-t5-large (bigger download, give it a minute)...


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.13GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## 5. Embedding Creation
Converts every chunk into a dense vector using a Sentence-Transformers model.

In [19]:
print("Loading embedding model...")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

chunk_embeddings = embed_model.encode(
    all_chunks,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True   # so we can use cosine similarity via inner product
)

print("Embeddings shape:", chunk_embeddings.shape)

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (12, 384)


## 6. Vector Database (FAISS)
Stores embeddings in a FAISS index for fast similarity search.

In [20]:
embedding_dim = chunk_embeddings.shape[1]

# Inner product on normalized vectors == cosine similarity
index = faiss.IndexFlatIP(embedding_dim)
index.add(chunk_embeddings)

print(f"FAISS index built with {index.ntotal} vectors (dim={embedding_dim}).")

FAISS index built with 12 vectors (dim=384).


## 7. Query Processing + Context Retrieval
Embeds the user's question and retrieves the most relevant chunks.

In [21]:
def retrieve_context(query, top_k=TOP_K):
    query_vec = embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, indices = index.search(query_vec, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        results.append({
            "text": all_chunks[idx],
            "score": float(score),
            "source": chunk_metadata[idx]["source"],
            "chunk_id": chunk_metadata[idx]["chunk_id"],
        })
    return results

# quick test
test_results = retrieve_context("What is RAG?")
for r in test_results:
    print(f"[{r['source']} | chunk {r['chunk_id']} | score {r['score']:.3f}]")
    print(r["text"][:200], "...\n")

[sample_rag_notes.txt | chunk 0 | score 0.523]
Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text generation. Instead of relying purely on a language model's parametric memory, RAG systems first searc ...

[sample_rag_notes.txt | chunk 2 | score 0.506]
odel produces the final answer using both the original question and the retrieved context. RAG systems are useful because they reduce hallucination, allow question answering over private or domain-spe ...

[sample_rag_notes.txt | chunk 1 | score 0.267]
hree main stages: retrieval, augmentation, and generation. In the retrieval stage, the user's query is converted into a vector embedding and compared against a vector database of document chunk embedd ...

[resume.pdf | chunk 3 | score 0.174]
ka, PostgreSQL GitHub • Built a distributed, containerized service focused on availability and scalability: rate limiting, caching, and duplicate-request handling under load • Debugged race conditions ...

## 8. Answer Generation
Uses a local, free Hugging Face model (`flan-t5-base`) to generate an answer grounded in the retrieved context.

In [24]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Loading generation model (this may take a minute the first time)...")
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL_NAME)

def build_prompt(question, context_chunks):
    context_text = "\n\n".join([c["text"] for c in context_chunks])
    prompt = (
        "Answer the question using ONLY the context below. "
        "If the answer is not in the context, say 'I don't have enough information.'\n\n"
        f"Context:\n{context_text}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )
    return prompt

def generate_answer(question, top_k=TOP_K):
    context_chunks = retrieve_context(question, top_k=top_k)
    prompt = build_prompt(question, context_chunks)
    input_ids = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).input_ids
    output_ids = gen_model.generate(input_ids, max_new_tokens=200)
    answer = gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return answer, context_chunks

Loading generation model (this may take a minute the first time)...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


## 9. End-to-End Demo
Ask questions about the ingested documents.

In [25]:
questions = [
    "What is Retrieval-Augmented Generation?",
    "What are the three main stages of a RAG pipeline?",
    "Name some popular vector databases mentioned in the document.",
    "Why does chunk size matter for retrieval quality?",
]

for q in questions:
    answer, sources = generate_answer(q)
    print("=" * 80)
    print("Q:", q)
    print("A:", answer)
    print("Sources used:", [f"{s['source']}#chunk{s['chunk_id']}" for s in sources])
    print()

Q: What is Retrieval-Augmented Generation?
A: combines information retrieval with text generation
Sources used: ['sample_rag_notes.txt#chunk0', 'sample_rag_notes.txt#chunk1', 'sample_rag_notes.txt#chunk2', 'sample_rag_notes.txt#chunk3']

Q: What are the three main stages of a RAG pipeline?
A: retrieval, augmentation, and generation
Sources used: ['sample_rag_notes.txt#chunk0', 'sample_rag_notes.txt#chunk1', 'sample_rag_notes.txt#chunk2', 'resume.pdf#chunk4']

Q: Name some popular vector databases mentioned in the document.
A: FAISS, Chroma, Pinecone, and Weaviate
Sources used: ['sample_rag_notes.txt#chunk2', 'sample_rag_notes.txt#chunk3', 'sample_rag_notes.txt#chunk1', 'resume.pdf#chunk3']

Q: Why does chunk size matter for retrieval quality?
A: Too large dilute relevance
Sources used: ['sample_rag_notes.txt#chunk1', 'sample_rag_notes.txt#chunk3', 'sample_rag_notes.txt#chunk0', 'sample_rag_notes.txt#chunk2']



## 10. Interactive Q&A
Run this cell and type your own questions (type `exit` to stop).

In [38]:
while True:
    user_q = input("Ask a question about your documents (or 'exit'): ")
    if user_q.strip().lower() == "exit":
        break
    ans, srcs = generate_answer(user_q)
    print("\nAnswer:", ans)
    print("Retrieved from:", [f"{s['source']}#chunk{s['chunk_id']}" for s in srcs])
    print("-" * 60)

Ask a question about your documents (or 'exit'): what is the name of the candidate?

Answer: Mahesh Tulshiram Shinde
Retrieved from: ['resume.pdf#chunk6', 'resume.pdf#chunk0', 'resume.pdf#chunk5', 'resume.pdf#chunk7']
------------------------------------------------------------
Ask a question about your documents (or 'exit'): projects?

Answer: Mahesh Tulshiram Shinde is a final-year Computer Science student with hands-on experience building and deploying backend services on Linux, working with Python and Java, containers, and public cloud infrastructure. Comfortable writing tests, debugging production issues, and collabo- rating with a team in an agile workflow. Technical Ski pm; GSSoC 2026 Project Admin • RestroHub – authentication bug fix • TermUI – terminal UI framework contributions Education B.Tech, Computer Engineering Expected 2027 Sanjivani College of Engineering, SPPU CGPA: 8.32/10 Diploma, Computer Technology 2024 Sanjivani K.B.P. Polytechnic, MSBTE 91.37% Achievements 416+ 

In [30]:
import os
sample_path = os.path.join(DOCS_FOLDER, "sample_rag_notes.txt")
if os.path.exists(sample_path):
    os.remove(sample_path)

documents = load_documents(DOCS_FOLDER)
print(f"Loaded {len(documents)} document(s).")

Loaded 1 document(s).


In [31]:
results = retrieve_context("What is the candidate's name?")
for r in results:
    print(r['source'], r['chunk_id'], '\n', r['text'][:300], '\n---')


resume.pdf 6 
 pm; GSSoC 2026 Project Admin • RestroHub – authentication bug fix • TermUI – terminal UI framework contributions Education B.Tech, Computer Engineering Expected 2027 Sanjivani College of Engineering, SPPU CGPA: 8.32/10 Diploma, Computer Technology 2024 Sanjivani K.B.P . Polytechnic, MSBTE 91.37% Ach 
---
sample_rag_notes.txt 1 
 hree main stages: retrieval, augmentation, and generation. In the retrieval stage, the user's query is converted into a vector embedding and compared against a vector database of document chunk embeddings using similarity search (commonly cosine similarity or L2 distance). In the augmentation stage, 
---
resume.pdf 0 
 Mahesh Tulshiram Shinde Nashik, Maharashtra — maheshshinde9100@gmail.com — +91-9529544681 github.com/maheshshinde9100 — linkedin.com/in/maheshshinde9100 — maheshshinde-dev.vercel.app Summary Final-year Computer Science student with hands-on experience building and deploying backend services on Linux 
---
sample_rag_notes.txt 2 
 od

In [37]:
print(generate_answer("What is the candidate's name?"))
print(generate_answer("What is the candidate's email and phone number?"))
print(generate_answer("What projects has this candidate worked on?"))

('Mahesh Tulshiram Shinde', [{'text': 'pm; GSSoC 2026 Project Admin • RestroHub – authentication bug fix • TermUI – terminal UI framework contributions Education B.Tech, Computer Engineering Expected 2027 Sanjivani College of Engineering, SPPU CGPA: 8.32/10 Diploma, Computer Technology 2024 Sanjivani K.B.P . Polytechnic, MSBTE 91.37% Achievements 416+ LeetCode problems solved — 3,238+ GitHub contributions — Google Cloud Skills Boost – 37 badges — AWS Educate – 8 badges', 'score': 0.16133180260658264, 'source': 'resume.pdf', 'chunk_id': 6}, {'text': 'Mahesh Tulshiram Shinde Nashik, Maharashtra — maheshshinde9100@gmail.com — +91-9529544681 github.com/maheshshinde9100 — linkedin.com/in/maheshshinde9100 — maheshshinde-dev.vercel.app Summary Final-year Computer Science student with hands-on experience building and deploying backend services on Linux, working with Python and Java, containers, and public cloud infrastructure. Comfortable writing tests, debugging production issues, and collabo

---

## 12. Conclusion

This notebook implements a full Retrieval-Augmented Generation pipeline:

**Ingestion → Chunking → Embedding → Vector Store (FAISS) → Retrieval → Generation**

Key takeaways:
- RAG grounds LLM answers in real documents, reducing hallucination.
- Retrieval quality (chunking + embedding model) directly affects answer quality.
- FAISS enables fast similarity search even over large document collections.
- The pipeline is modular — you can swap the embedding model, vector store, or generator independently.

To use with your own data: drop PDFs/TXT files into the `documents/` folder and re-run from Section 3 onward.

---

## Submitted By

**Name:** Mahesh Shinde

**College:** Sanjivani College of Engineering, Kopargaon

**Internship:** CEI Internship – Data Science
